# AtlasIR — full SciFact BM25 / E5 / hybrid comparison

Attach only the private `atlasir-scifact-comparison-input.zip` dataset. Enable Internet and T4, then run all cells. This evaluates **5,183 documents, 647 training queries and 162 validation queries**, with no final-test judgments. It produces fixed-method metrics and query-level routing opportunity analysis, not an adaptive-gain claim.

One download at the end contains the results, source, checkpoints, model and vector objects. Stop the GPU after downloading. See `docs/DAY08-COMPARISON.md`.


In [ ]:
import base64
import hashlib
import json
import os
import subprocess
import sys
import tarfile
import uuid
from pathlib import Path

# These optional roots allow the same cell to be exercised by local fixture tests.
input_root = Path(globals().get("ATLASIR_INPUT_ROOT", "/kaggle/input"))
working_root = Path(globals().get("ATLASIR_WORKING_ROOT", "/kaggle/working"))
candidates = list(input_root.rglob("launcher-input.json"))
assert len(candidates) == 1, "Attach exactly one AtlasIR input; remove the old dataset first"
UPLOADED = candidates[0].parent
manifest = json.loads(candidates[0].read_bytes())
assert manifest["schema_version"] in (1, 2), "Unsupported launcher input version"
assert manifest["job"] == "scifact-fixed-comparison-v1", "Attach the comparison input"
encoded = set(manifest.get("base64_files", []))
WORK = working_root / ("atlasir-comparison-" + uuid.uuid4().hex[:12])
INPUT = WORK / "input"
INPUT.mkdir(parents=True)
for name, digest in manifest["sha256"].items():
    relative = Path(name)
    assert not relative.is_absolute() and ".." not in relative.parts
    wire_name = name + ".b64" if name in encoded else name
    path = UPLOADED / wire_name
    assert path.resolve().is_relative_to(UPLOADED.resolve())
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {wire_name}. Kaggle may have extracted a legacy archive. "
            "Attach atlasir-comparison-input.zip and use the updated notebook."
        )
    content = path.read_bytes()
    if name in encoded:
        content = base64.b64decode(content, validate=True)
    assert hashlib.sha256(content).hexdigest() == digest, f"Input hash mismatch: {name}"
    destination = INPUT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(content)
(INPUT / "launcher-input.json").write_bytes(candidates[0].read_bytes())
SOURCE = WORK / "source"
SOURCE.mkdir()
with tarfile.open(INPUT / "source.tar.gz", "r:gz") as archive:
    seen = set()
    total = 0
    for member in archive:
        relative = Path(member.name)
        total += member.size
        assert member.isfile() and not relative.is_absolute()
        assert ".." not in relative.parts and "\\" not in member.name
        assert member.name not in seen and total <= 128 * 1024 * 1024
        seen.add(member.name)
        destination = SOURCE / relative
        assert destination.resolve().is_relative_to(SOURCE.resolve())
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.extractfile(member) as stream:
            destination.write_bytes(stream.read())
print(
    {
        "reference": manifest["job"],
        "source_sha256": manifest["sha256"]["source.tar.gz"],
        "work": str(WORK),
        "input_checks": "passed",
    }
)

In [ ]:
# Install only the pinned launcher tool; project packages stay isolated.
TOOLS = WORK / "tools"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--target",
        str(TOOLS),
        "uv==" + manifest["uv_version"],
    ],
    check=True,
)
# The uv wheel installs its executable below the target's bin directory.
UV = TOOLS / "bin" / "uv"
assert UV.is_file(), "Pinned uv executable missing from tool target"
ENV = os.environ.copy()
ENV.pop("PYTHONPATH", None)
ENV["UV_CACHE_DIR"] = str(WORK / "uv-cache")
ENV["UV_PROJECT_ENVIRONMENT"] = str(WORK / "project-env")
ENV["UV_PYTHON"] = manifest["python_profile"]
BASE = [
    str(UV),
    "run",
    "--project",
    str(SOURCE),
    "--python",
    manifest["python_profile"],
    "--locked",
    "--no-dev",
    "--extra",
    "dense",
    "--extra",
    "search",
]
subprocess.run(BASE + ["python", "-m", "atlasir", "smoke"], env=ENV, cwd=SOURCE, check=True)

In [ ]:
# This probes the actual locked neural environment, not the notebook kernel.
ENV["HF_HOME"] = str(WORK / "model-cache")
ENV["HF_HUB_DISABLE_XET"] = "1"
ENV["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
ENV["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run(
    BASE
    + [
        "python",
        str(SOURCE / "scripts/runtime_probe.py"),
        "--require-gpu",
        "--output",
        str(WORK / "dense-runtime.json"),
    ],
    env=ENV,
    cwd=SOURCE,
    check=True,
)

In [ ]:
# Rerun this cell (without restarting the workspace) to reuse completed shards.
RESULT = WORK / ("comparison-result-" + uuid.uuid4().hex[:12] + ".json")
command = BASE + [
    "python",
    str(SOURCE / "scripts/run_comparison.py"),
    "--data",
    str(INPUT / "data"),
    "--selection",
    str(INPUT / "selection.json"),
    "--device",
    "cuda:0",
    "--source-root",
    str(SOURCE),
    "--root",
    str(WORK / "output"),
    "--result-json",
    str(RESULT),
]
subprocess.run(command, env=ENV, cwd=SOURCE, check=True)
result = json.loads(RESULT.read_bytes())
EXPORT = Path(result["session_exports"])
subprocess.run(
    BASE + ["python", str(SOURCE / "scripts/run_comparison.py"), "--reevaluate", result["path"]],
    env=ENV,
    cwd=SOURCE,
    check=True,
)
summary = json.loads((EXPORT / "results.json").read_bytes())
for role, methods in summary["metrics"].items():
    print(role)
    for method, metrics in methods.items():
        print(method, metrics)
analysis = json.loads((Path(result["path"]) / "opportunity.json").read_bytes())
print("Validation descriptive oracle gap:", analysis["validation"]["oracle_gap"])

In [ ]:
import zipfile

from IPython.display import FileLink, display

DOWNLOAD = working_root / ("atlasir-scifact-results-" + result["run_id"] + ".zip")
with zipfile.ZipFile(DOWNLOAD, "x", compression=zipfile.ZIP_STORED) as archive:
    for path in sorted(EXPORT.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(EXPORT))
    archive.write(WORK / "dense-runtime.json", "dense-runtime.json")
    archive.write(INPUT / "launcher-input.json", "launcher-input.json")
    archive.write(INPUT / "SCIFACT-ATTRIBUTION.md", "SCIFACT-ATTRIBUTION.md")
    archive.writestr("session-result.json", json.dumps(result, sort_keys=True, indent=2))
print(DOWNLOAD, "SHA256:", hashlib.sha256(DOWNLOAD.read_bytes()).hexdigest())
display(FileLink(DOWNLOAD.name))
print("Download this one ZIP, keep a Drive copy, then stop the GPU session.")